In [3]:
from pathlib import Path
import re
import numpy as np
import pandas as pd


# ============================================================
# Overlap diagnostics for canonical query folders
#
# Expected structure:
# C:/Python/CSUREMM/data_events/
#     weather/
#         chunks/
#             gt_weather_2022-01-01_2022-09-27.csv
#             ...
#
# Output:
#   C:/Python/CSUREMM/data_events/overlap_diagnostics.csv
# ============================================================

DATA_DIR = Path(r"C:/Python/CSUREMM/data_events")
OUTPUT_FILE = DATA_DIR / "overlap_diagnostics.csv"

MIN_OVERLAP_DAYS = 10
LOW_CORR_THRESHOLD = 0.85
HIGH_RATIO_CV_THRESHOLD = 0.25
MEDIAN_RATIO_LOW = 0.75
MEDIAN_RATIO_HIGH = 1.33
MEAN_ABS_PCT_DIFF_THRESHOLD = 25.0


def extract_date_range(path: Path):
    dates = re.findall(
        r"\d{4}-\d{2}-\d{2}",
        path.stem
    )

    if len(dates) >= 2:
        return pd.to_datetime(dates[-2]), pd.to_datetime(dates[-1])

    return pd.NaT, pd.NaT


def read_gt_chunk(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        parse_dates=["Time"]
    )

    value_cols = [
        c for c in df.columns
        if c != "Time"
    ]

    if len(value_cols) != 1:
        raise ValueError(
            f"{path} has {len(value_cols)} value columns: {value_cols}"
        )

    value_col = value_cols[0]

    df = (
        df[["Time", value_col]]
        .rename(columns={value_col: "value"})
        .sort_values("Time")
        .drop_duplicates("Time", keep="first")
        .reset_index(drop=True)
    )

    df["value"] = pd.to_numeric(
        df["value"],
        errors="coerce"
    )

    return df


def diagnose_pair(query_folder: Path, left_file: Path, right_file: Path) -> dict:

    left = read_gt_chunk(left_file).rename(
        columns={"value": "left_value"}
    )

    right = read_gt_chunk(right_file).rename(
        columns={"value": "right_value"}
    )

    overlap = pd.merge(
        left,
        right,
        on="Time",
        how="inner"
    ).dropna(
        subset=["left_value", "right_value"]
    )

    valid_ratio = overlap[
        (overlap["left_value"] > 0) &
        (overlap["right_value"] > 0)
    ].copy()

    left_std = overlap["left_value"].std()
    right_std = overlap["right_value"].std()

    if len(overlap) >= 2 and left_std > 0 and right_std > 0:
        overlap_corr = overlap["left_value"].corr(
            overlap["right_value"]
        )
    else:
        overlap_corr = np.nan

    if len(overlap) >= 2 and (
    overlap["left_value"].std() == 0 or
    overlap["right_value"].std() == 0
    ):
        flags.append("constant_overlap_series")

    if len(valid_ratio) > 0:
        ratios = (
            valid_ratio["right_value"]
            / valid_ratio["left_value"]
        )

        median_ratio = ratios.median()
        mean_ratio = ratios.mean()
        ratio_std = ratios.std()
        ratio_cv = (
            ratio_std / mean_ratio
            if mean_ratio != 0
            else np.nan
        )

        abs_pct_diff = (
            (valid_ratio["right_value"] - valid_ratio["left_value"]).abs()
            / valid_ratio[["left_value", "right_value"]].mean(axis=1)
            * 100
        )

        mean_abs_pct_diff = abs_pct_diff.mean()
        median_abs_pct_diff = abs_pct_diff.median()

    else:
        median_ratio = np.nan
        mean_ratio = np.nan
        ratio_std = np.nan
        ratio_cv = np.nan
        mean_abs_pct_diff = np.nan
        median_abs_pct_diff = np.nan

    left_start, left_end = extract_date_range(left_file)
    right_start, right_end = extract_date_range(right_file)

    flags = []

    if len(overlap) < MIN_OVERLAP_DAYS:
        flags.append("short_overlap")

    if pd.notna(overlap_corr) and overlap_corr < LOW_CORR_THRESHOLD:
        flags.append("low_overlap_correlation")

    if pd.notna(ratio_cv) and ratio_cv > HIGH_RATIO_CV_THRESHOLD:
        flags.append("unstable_ratio")

    if pd.notna(median_ratio) and (
        median_ratio < MEDIAN_RATIO_LOW or
        median_ratio > MEDIAN_RATIO_HIGH
    ):
        flags.append("large_scale_shift")

    if pd.notna(mean_abs_pct_diff) and (
        mean_abs_pct_diff > MEAN_ABS_PCT_DIFF_THRESHOLD
    ):
        flags.append("large_average_difference")

    if not flags:
        flags.append("ok")

    return {
        "query": query_folder.name,
        "left_file": left_file.name,
        "right_file": right_file.name,
        "left_start": left_start,
        "left_end": left_end,
        "right_start": right_start,
        "right_end": right_end,
        "overlap_start": overlap["Time"].min() if len(overlap) else pd.NaT,
        "overlap_end": overlap["Time"].max() if len(overlap) else pd.NaT,
        "overlap_days": len(overlap),
        "valid_ratio_days": len(valid_ratio),
        "overlap_corr": overlap_corr,
        "median_ratio_right_over_left": median_ratio,
        "mean_ratio_right_over_left": mean_ratio,
        "ratio_std": ratio_std,
        "ratio_cv": ratio_cv,
        "mean_abs_pct_diff": mean_abs_pct_diff,
        "median_abs_pct_diff": median_abs_pct_diff,
        "abnormal_flag": int(flags != ["ok"]),
        "flag_reason": ";".join(flags)
    }


rows = []
failed = []

query_folders = [
    p for p in DATA_DIR.iterdir()
    if p.is_dir()
]

for query_folder in sorted(query_folders):

    chunks_dir = query_folder / "chunks"

    if not chunks_dir.exists():
        continue

    chunk_files = sorted(
        chunks_dir.glob("gt_*.csv"),
        key=lambda p: extract_date_range(p)[0]
    )

    if len(chunk_files) < 2:
        continue

    for i in range(len(chunk_files) - 1):

        left_file = chunk_files[i]
        right_file = chunk_files[i + 1]

        try:
            rows.append(
                diagnose_pair(
                    query_folder=query_folder,
                    left_file=left_file,
                    right_file=right_file
                )
            )

        except Exception as e:
            failed.append({
                "query": query_folder.name,
                "left_file": left_file.name,
                "right_file": right_file.name,
                "error": str(e)
            })


diagnostics = pd.DataFrame(rows)

if len(diagnostics) > 0:
    diagnostics = diagnostics.sort_values(
        [
            "abnormal_flag",
            "query",
            "left_start",
            "right_start"
        ],
        ascending=[
            False,
            True,
            True,
            True
        ]
    )

diagnostics.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"saved: {OUTPUT_FILE}")
print(f"diagnostic rows: {len(diagnostics)}")

if len(diagnostics) > 0:
    print(f"abnormal rows: {diagnostics['abnormal_flag'].sum()}")

if failed:
    failed_file = DATA_DIR / "overlap_diagnostics_failed.csv"

    pd.DataFrame(failed).to_csv(
        failed_file,
        index=False
    )

    print(f"failed pairs saved: {failed_file}")

saved: C:\Python\CSUREMM\data_events\overlap_diagnostics.csv
diagnostic rows: 1054
abnormal rows: 621
failed pairs saved: C:\Python\CSUREMM\data_events\overlap_diagnostics_failed.csv
